In [2]:
import numpy as np
import nibabel as nib
import pandas as pd
from nibabel import Nifti1Image
from nibabel.processing import resample_to_output

# Load RSA results
rsa_data = np.load("searchlight_rsa.npy")  # shape: (n_subjects, 575, 56949)
n_subjects, n_timepoints, n_voxels = rsa_data.shape

# Load voxel index map (voxel indices into 53x63x46 volume)
voxel_center_id = pd.read_csv(
    r"N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\outputs\voxel_center_id.csv",
    header=0,
    index_col=0
)
voxel_indices = np.squeeze(np.array(voxel_center_id))  # shape: (56949,)

tmp_image_path = 'N:/Experimental_Data/yujunchen/projects/IAPS_fMRI_RSA/fMRI_singletrial_betas/nifti/Nt1.img'
tmp_img = nib.load(tmp_image_path)

# Define 3D volume shape
x, y, z = 53, 63, 46
volume_shape = (x, y, z)

# Loop over subjects
for subj_idx in range(n_subjects):
    print(f"Processing subject {subj_idx+1}/{n_subjects}")

    # Initialize 4D array (x, y, z, time)
    rsa_4d = np.full((x * y * z, n_timepoints), np.nan)

    # Assign each timepoint's RSA values to corresponding voxels
    for t in range(n_timepoints):
        rsa_4d[voxel_indices, t] = rsa_data[subj_idx, t, :]

    # Reshape to 4D volume (x, y, z, time)
    rsa_4d_volume = rsa_4d.reshape((x, y, z, n_timepoints))

    # Create NIfTI
    rsa_img = nib.Nifti1Image(rsa_4d_volume, affine=tmp_img.affine, header=tmp_img.header)

    # Save to disk
    nib.save(rsa_img, f"rsa_subject{subj_idx+1:02d}.nii.gz")


Processing subject 1/20
Processing subject 2/20
Processing subject 3/20
Processing subject 4/20
Processing subject 5/20
Processing subject 6/20
Processing subject 7/20
Processing subject 8/20
Processing subject 9/20
Processing subject 10/20
Processing subject 11/20
Processing subject 12/20
Processing subject 13/20
Processing subject 14/20
Processing subject 15/20
Processing subject 16/20
Processing subject 17/20
Processing subject 18/20
Processing subject 19/20
Processing subject 20/20


In [4]:
import numpy as np
import nibabel as nib

# List of subject filenames, strictly rsa_subject01.nii.gz to rsa_subject20.nii.gz
n_subjects = 20
file_list = [f"rsa_subject{i:02d}.nii.gz" for i in range(1, n_subjects + 1)]

# Load first file for shape/affine
img0 = nib.load(file_list[0])
data_shape = img0.shape  # (53, 63, 46, 575)

# Pre-allocate
all_data = np.zeros((n_subjects, *data_shape), dtype=np.float32)

# Load each subject's data
for idx, fname in enumerate(file_list):
    img = nib.load(fname)
    all_data[idx] = img.get_fdata()

# Nanmean over subjects
mean_data = np.nanmean(all_data, axis=0)

# Save mean image
mean_img = nib.Nifti1Image(mean_data, affine=img0.affine, header=img0.header)
nib.save(mean_img, "rsa_subject_mean.nii.gz")

print("Saved rsa_subject_mean.nii.gz")


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_20336\1419527549.py:21: RuntimeWarning: Mean of empty slice
  mean_data = np.nanmean(all_data, axis=0)


Saved rsa_subject_mean.nii.gz


In [6]:
import numpy as np
import nibabel as nib
from nilearn.image import resample_img

img = nib.load("rsa_subject_mean.nii.gz")

# choose target voxel size (2 mm is a good balance)
vx = np.array([2.0, 2.0, 2.0])

# current spatial zooms and FOV
zooms = nib.affines.voxel_sizes(img.affine)[:3]
fov = np.array(img.shape[:3]) * zooms

# target grid shape to cover the same FOV
target_shape = tuple(np.ceil(fov / vx).astype(int))

# build a proper 4×4 target affine:
A = img.affine[:3, :3]                           # current orientation * voxel sizes
new_A = A @ np.diag(vx / zooms)                  # keep orientation, update voxel sizes
target_affine = np.eye(4)
target_affine[:3, :3] = new_A
target_affine[:3,  3] = img.affine[:3, 3]        # keep origin

# resample in space (not time)
hi = resample_img(
    img,
    target_affine=target_affine,
    target_shape=target_shape,
    interpolation="continuous",
)

# (optional) cast to float32 to keep file size manageable
hi32 = nib.Nifti1Image(hi.get_fdata(dtype=np.float32), hi.affine, hi.header)
hi32.header.set_data_dtype(np.float32)
nib.save(hi32, "rsa_subject_mean_sm4mm_2mm.nii.gz")


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_24004\1520210593.py:25: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_24004\1520210593.py:25: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_24004\1520210593.py:25: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a bad thing as they make resampling ill-defined and much slower.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_24004\1520210593.py:25: RuntimeWarning: NaNs or infinite values are present in the data passed to resample. This is a b

## mesh

In [10]:
"""
Make a smooth mesh + vertex time series from a 4D searchlight volume.

Inputs
  - rsa_subject_mean.nii.gz  (4D: X×Y×Z×T)

Outputs
  - rsa_subject_mean_sm4mm_1mm.nii.gz      (optional, smoothed+1mm resampled 4D)
  - rsa_subject_mean_mesh.surf.gii         (triangle mesh in mm/world coords)
  - rsa_subject_mean_vertex_ts.func.gii    (N_vertices × T, for Movie mode)

Requires: numpy, nibabel, nilearn, scipy, scikit-image
"""

import numpy as np
import numpy.linalg as npl
import nibabel as nib

from nilearn.image import smooth_img, resample_img
from nibabel.affines import apply_affine
from nibabel.gifti import GiftiImage, GiftiDataArray

from scipy.ndimage import gaussian_filter, label, map_coordinates
from skimage.measure import marching_cubes

# ======================= SETTINGS =======================
IN_4D = "rsa_subject_mean.nii.gz"          # your 4D searchlight
OUT_4D_HI = "rsa_subject_mean_sm4mm_1mm.nii.gz"

# spatial smoothing (mm) before resampling – helps reduce blockiness
SMOOTH_FWHM_MM = 4.0

# target voxel size for better 3D rendering
VOX_MM = (1.0, 1.0, 1.0)

# Mesh construction from a 3D aggregate map:
AGG_METHOD = "absmean"   # 'mean' | 'absmean' | 'max'
PERCENTILE = 97.0        # top X% voxels kept for the mesh region (use THRESH_VAL to override)
THRESH_VAL = None        # absolute threshold; set a float to use instead of percentile

# clean up: drop very small components (in voxels)
MIN_CLUSTER_VOX = 500

# smooth the binary mask a bit before meshing (rounder surface)
MASK_SMOOTH_FWHM_MM = 2.0
# ========================================================


def resample_4d_to_1mm(img_4d, voxel_sizes):
    """Resample a 4D image to given voxel sizes, preserving orientation & origin."""
    zooms = nib.affines.voxel_sizes(img_4d.affine)[:3]
    fov = np.array(img_4d.shape[:3]) * zooms
    target_shape = tuple(np.ceil(fov / np.array(voxel_sizes)).astype(int))

    A = img_4d.affine[:3, :3]
    new_A = A @ np.diag(np.array(voxel_sizes) / zooms)
    target_affine = np.eye(4)
    target_affine[:3, :3] = new_A
    target_affine[:3,  3] = img_4d.affine[:3, 3]

    hi = resample_img(
        img_4d,
        target_affine=target_affine,
        target_shape=target_shape,
        interpolation="continuous",   # cubic
        copy=True,
    )
    return hi


def aggregate_3d(vol4d, how="absmean"):
    if vol4d.ndim == 3:
        return vol4d
    if how == "mean":
        return np.nanmean(vol4d, axis=3)
    if how == "absmean":
        return np.nanmean(np.abs(vol4d), axis=3)
    if how == "max":
        return np.nanmax(vol4d, axis=3)
    raise ValueError("Unknown AGG_METHOD")


def make_mask(agg3d, percentile=97.0, thr_val=None, min_cluster=500, affine=None, mask_smooth_fwhm=2.0):
    vol = np.nan_to_num(agg3d, nan=0.0)
    vals = vol[vol != 0]
    thr = float(thr_val) if (thr_val is not None) else (np.percentile(vals, percentile) if vals.size else 0.0)
    mask = vol >= thr

    # remove tiny islands
    if mask.any():
        lab, n = label(mask)
        if n > 0:
            counts = np.bincount(lab.ravel())
            keep = np.where(counts >= max(int(min_cluster), 1))[0]
            keep = keep[keep != 0]  # drop background
            mask = np.isin(lab, keep)

    # optional smoothing for a rounder surface
    if mask_smooth_fwhm and mask_smooth_fwhm > 0:
        vx = nib.affines.voxel_sizes(affine)[:3] if affine is not None else (1.0, 1.0, 1.0)
        sigma_vox = (mask_smooth_fwhm / 2.3548) / np.array(vx)
        smooth = gaussian_filter(mask.astype(np.float32), sigma=sigma_vox, mode="nearest")
        vol_for_surface = smooth
        level = 0.5
    else:
        vol_for_surface = mask.astype(np.float32)
        level = 0.5

    return vol_for_surface, level


def mesh_from_volume(vol_for_surface, level, affine):
    """
    marching_cubes expects array ordered as (Z,Y,X). NIfTI data is (X,Y,Z).
    We will transform the resulting vertex voxel coords to world (mm) with affine.
    """
    arr_zyx = np.transpose(vol_for_surface, (2, 1, 0))
    verts_zyx, faces, _, _ = marching_cubes(arr_zyx, level=level, allow_degenerate=False)
    verts_xyz_vox = verts_zyx[:, [2, 1, 0]]  # to (X,Y,Z)
    verts_mm = apply_affine(affine, verts_xyz_vox)
    return verts_mm.astype(np.float32), faces.astype(np.int32)


def save_surf_gii(verts_mm, faces, out_path):
    gii = GiftiImage()
    gii.add_gifti_data_array(GiftiDataArray(verts_mm, intent='NIFTI_INTENT_POINTSET'))
    gii.add_gifti_data_array(GiftiDataArray(faces, intent='NIFTI_INTENT_TRIANGLE'))
    nib.save(gii, out_path)


def sample_vertex_timeseries(verts_mm, img4d, out_func_path):
    """
    Sample the 4D volume at mesh vertex locations (trilinear) → (N_vertices × T)
    """
    data = img4d.get_fdata()
    aff = img4d.affine
    iaff = npl.inv(aff)

    hom = np.c_[verts_mm, np.ones(len(verts_mm))]
    xyz_vox = (hom @ iaff.T)[:, :3].T  # shape (3, N)

    if data.ndim == 3:
        data = data[..., np.newaxis]
    T = data.shape[3]
    vals = np.empty((verts_mm.shape[0], T), dtype=np.float32)

    # sample each time point
    for t in range(T):
        vals[:, t] = map_coordinates(data[..., t], xyz_vox, order=1, mode="nearest")

    gii = GiftiImage()
    gii.add_gifti_data_array(GiftiDataArray(vals, intent='NIFTI_INTENT_TIME_SERIES'))
    nib.save(gii, out_func_path)
    return vals.shape


def main():
    # ---- load 4D, smooth, resample to 1mm (for nicer rendering) ----
    img4d = nib.load(IN_4D)

    if SMOOTH_FWHM_MM and SMOOTH_FWHM_MM > 0:
        img4d = smooth_img(img4d, fwhm=SMOOTH_FWHM_MM)

    img4d_hi = resample_4d_to_1mm(img4d, VOX_MM)

    # save compact float32
    hi32 = nib.Nifti1Image(img4d_hi.get_fdata(dtype=np.float32), img4d_hi.affine, img4d_hi.header)
    hi32.header.set_data_dtype(np.float32)
    nib.save(hi32, OUT_4D_HI)
    print(f"[+] Saved: {OUT_4D_HI}  shape={hi32.shape}")

    # ---- aggregate to 3D for meshing ----
    agg3d = aggregate_3d(hi32.get_fdata(), how=AGG_METHOD)

    # ---- make mask volume for surface extraction ----
    vol_for_surface, level = make_mask(
        agg3d,
        percentile=PERCENTILE,
        thr_val=THRESH_VAL,
        min_cluster=MIN_CLUSTER_VOX,
        affine=hi32.affine,
        mask_smooth_fwhm=MASK_SMOOTH_FWHM_MM,
    )

    if not np.any(vol_for_surface > 0):
        raise RuntimeError("Mask is empty after thresholding/cleanup. Relax thresholds.")

    # ---- marching cubes → verts (mm), faces ----
    verts_mm, faces = mesh_from_volume(vol_for_surface, level, hi32.affine)
    print(f"[+] Mesh: {verts_mm.shape[0]} vertices, {faces.shape[0]} faces")

    # ---- save surface mesh ----
    surf_path = "rsa_subject_mean_mesh.surf.gii"
    save_surf_gii(verts_mm, faces, surf_path)
    print(f"[+] Saved mesh: {surf_path}")

    # ---- sample vertex time series from 4D (the resampled/smoothed one) ----
    func_path = "rsa_subject_mean_vertex_ts.func.gii"
    ts_shape = sample_vertex_timeseries(verts_mm, hi32, func_path)
    print(f"[+] Saved vertex time series: {func_path}  shape={ts_shape}")


if __name__ == "__main__":
    main()


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_24004\2626829177.py:61: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  hi = resample_img(
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_24004\2626829177.py:61: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  hi = resample_img(


[+] Saved: rsa_subject_mean_sm4mm_1mm.nii.gz  shape=(159, 189, 138, 575)
[+] Mesh: 32874 vertices, 65736 faces
[+] Saved mesh: rsa_subject_mean_mesh.surf.gii
[+] Saved vertex time series: rsa_subject_mean_vertex_ts.func.gii  shape=(32874, 575)


## Resample and smoothing

In [15]:
#!/usr/bin/env python
"""
All-in-One Smooth Brain 3D Visualization Pipeline for Jupyter
Run this entire cell to transform your blocky cube into a smooth brain surface
"""

# ============================================================================
# IMPORTS
# ============================================================================
import numpy as np
import nibabel as nib
from scipy.ndimage import binary_fill_holes, gaussian_filter
import warnings
warnings.filterwarnings('ignore')

# Optional imports (will check if available)
try:
    from nilearn.image import resample_img, smooth_img
    NILEARN_AVAILABLE = True
except ImportError:
    NILEARN_AVAILABLE = False
    print("Note: nilearn not fully available, some features may be limited")

try:
    from skimage import measure
    SKIMAGE_AVAILABLE = True
except ImportError:
    SKIMAGE_AVAILABLE = False
    print("Note: scikit-image not available, surface mesh creation will be skipped")

# ============================================================================
# MAIN SMOOTHING FUNCTION
# ============================================================================

def create_smooth_brain_surface(input_file, output_file, smooth_fwhm=6.0, threshold_percentile=20):
    """
    Create a smooth brain surface for 3D visualization in FSLeyes
    
    Parameters:
    -----------
    input_file : str
        Path to input nifti file
    output_file : str
        Path to output nifti file
    smooth_fwhm : float
        Smoothing kernel size in mm (higher = smoother)
    threshold_percentile : float
        Percentile threshold to remove background noise
    """
    
    print(f"\n{'='*60}")
    print(f"Processing: {input_file}")
    print(f"{'='*60}")
    
    # Load data
    print("1. Loading data...")
    img = nib.load(input_file)
    data = img.get_fdata()
    original_shape = data.shape
    print(f"   Data shape: {original_shape}")
    
    # Remove NaN values
    data = np.nan_to_num(data, nan=0.0)
    
    # Calculate threshold
    print("2. Calculating threshold...")
    non_zero_data = data[data != 0]
    if len(non_zero_data) > 0:
        threshold = np.percentile(np.abs(non_zero_data), threshold_percentile)
    else:
        threshold = 0.001
    print(f"   Threshold: {threshold:.6f}")
    
    # Process 4D or 3D data
    print("3. Creating brain mask and smoothing...")
    if len(data.shape) == 4:
        print(f"   Processing 4D data with {data.shape[3]} timepoints...")
        n_timepoints = data.shape[3]
        smoothed_data = np.zeros_like(data)
        
        # Process each timepoint
        for t in range(n_timepoints):
            vol = data[:, :, :, t]
            
            # Create mask
            vol_mask = np.abs(vol) > threshold
            vol_mask = binary_fill_holes(vol_mask)
            vol_mask = gaussian_filter(vol_mask.astype(float), sigma=1.0)
            vol_mask = vol_mask > 0.3
            
            # Apply smoothing
            smoothed_vol = gaussian_filter(vol, sigma=smooth_fwhm/2.355)
            smoothed_vol = smoothed_vol * vol_mask
            smoothed_data[:, :, :, t] = smoothed_vol
            
            # Progress indicator
            if t % 100 == 0 or t == n_timepoints - 1:
                print(f"   Processed timepoint {t+1}/{n_timepoints}", end='\r')
        print()  # New line after progress
        
        # Create envelope from mean
        mean_data = np.mean(np.abs(smoothed_data), axis=3)
    else:
        print("   Processing 3D data...")
        # Process 3D data
        mask_data = np.abs(data) > threshold
        mask_data = binary_fill_holes(mask_data)
        mask_data = gaussian_filter(mask_data.astype(float), sigma=1.0)
        mask_data = mask_data > 0.3
        
        smoothed_data = gaussian_filter(data, sigma=smooth_fwhm/2.355)
        smoothed_data = smoothed_data * mask_data
        mean_data = np.abs(smoothed_data)
    
    # Create final brain mask
    print("4. Creating smooth brain envelope...")
    brain_mask = mean_data > np.percentile(mean_data[mean_data > 0], 5)
    brain_mask = binary_fill_holes(brain_mask)
    brain_mask = gaussian_filter(brain_mask.astype(float), sigma=2.0)
    brain_mask = brain_mask > 0.2
    
    # Apply final mask
    if len(smoothed_data.shape) == 4:
        for t in range(smoothed_data.shape[3]):
            smoothed_data[:, :, :, t] = smoothed_data[:, :, :, t] * brain_mask
    else:
        smoothed_data = smoothed_data * brain_mask
    
    # Save result
    print("5. Saving smooth brain...")
    smooth_img = nib.Nifti1Image(smoothed_data.astype(np.float32), img.affine, img.header)
    smooth_img.header.set_data_dtype(np.float32)
    nib.save(smooth_img, output_file)
    
    # Report file size
    import os
    file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
    print(f"   Saved: {output_file} ({file_size:.1f} MB)")
    
    return smooth_img

# ============================================================================
# HIGH RESOLUTION VERSION (OPTIONAL)
# ============================================================================

def create_high_res_smooth_brain(input_file, output_file, target_resolution=1.5):
    """
    Create a high-resolution smooth brain with resampling
    """
    if not NILEARN_AVAILABLE:
        print("Skipping high-res version (requires nilearn)")
        return None
    
    print(f"\n{'='*60}")
    print("Creating high-resolution smooth brain...")
    print(f"{'='*60}")
    
    # First create smooth version
    print("1. Creating initial smooth version...")
    temp_smooth = "temp_smooth.nii.gz"
    smooth_brain = create_smooth_brain_surface(
        input_file, 
        temp_smooth,
        smooth_fwhm=6.0,
        threshold_percentile=15
    )
    
    # Resample to higher resolution
    print(f"2. Resampling to {target_resolution}mm resolution...")
    voxel_sizes = nib.affines.voxel_sizes(smooth_brain.affine)[:3]
    
    # Calculate new shape
    target_affine = smooth_brain.affine.copy()
    scale_factors = voxel_sizes / target_resolution
    target_affine[:3, :3] = smooth_brain.affine[:3, :3] / scale_factors[:, np.newaxis]
    
    current_shape = smooth_brain.shape[:3]
    target_shape = tuple(int(s * sf) for s, sf in zip(current_shape, scale_factors))
    
    print(f"   Original voxel size: {voxel_sizes}")
    print(f"   Target voxel size: [{target_resolution}, {target_resolution}, {target_resolution}]")
    print(f"   Original shape: {current_shape}")
    print(f"   Target shape: {target_shape}")
    
    # Resample
    resampled = resample_img(
        smooth_brain,
        target_affine=target_affine,
        target_shape=target_shape,
        interpolation='continuous'
    )
    
    # Apply final smoothing
    print("3. Applying final smoothing at high resolution...")
    final_smooth = smooth_img(resampled, fwhm=2.0)
    
    # Save
    nib.save(final_smooth, output_file)
    
    # Report file size
    import os
    file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
    print(f"   Saved: {output_file} ({file_size:.1f} MB)")
    
    # Clean up temp file
    if os.path.exists(temp_smooth):
        os.remove(temp_smooth)
    
    return final_smooth

# ============================================================================
# SURFACE MESH CREATION (OPTIONAL)
# ============================================================================

def create_surface_mesh(volume_file, output_prefix, threshold=0.01):
    """
    Create a surface mesh from volume data
    """
    if not SKIMAGE_AVAILABLE:
        print("\nSkipping surface mesh (requires scikit-image)")
        print("Install with: pip install scikit-image")
        return None
    
    from nibabel import gifti
    
    print(f"\n{'='*60}")
    print("Creating surface mesh...")
    print(f"{'='*60}")
    
    # Load volume
    print("1. Loading volume for surface extraction...")
    img = nib.load(volume_file)
    data = img.get_fdata()
    
    # Use mean if 4D
    if len(data.shape) == 4:
        print("   4D data detected, using mean for surface...")
        surface_data = np.mean(np.abs(data), axis=3)
    else:
        surface_data = np.abs(data)
    
    # Apply threshold
    surface_data[surface_data < threshold] = 0
    
    # Extract surface
    print("2. Extracting surface mesh (this may take a moment)...")
    verts, faces, normals, values = measure.marching_cubes(
        surface_data, 
        level=threshold,
        spacing=nib.affines.voxel_sizes(img.affine)[:3]
    )
    
    print(f"   Vertices: {len(verts):,}")
    print(f"   Faces: {len(faces):,}")
    
    # Transform to world coordinates
    verts_homogeneous = np.column_stack([verts, np.ones(len(verts))])
    verts_world = verts_homogeneous.dot(img.affine.T)[:, :3]
    
    # Create GIFTI
    print("3. Creating GIFTI surface file...")
    coords = gifti.GiftiDataArray(
        data=verts_world.astype(np.float32),
        intent='NIFTI_INTENT_POINTSET',
        datatype='NIFTI_TYPE_FLOAT32'
    )
    
    faces_array = gifti.GiftiDataArray(
        data=faces.astype(np.int32),
        intent='NIFTI_INTENT_TRIANGLE',
        datatype='NIFTI_TYPE_INT32'
    )
    
    gii = gifti.GiftiImage(darrays=[coords, faces_array])
    
    surface_file = f"{output_prefix}_surface.gii"
    nib.save(gii, surface_file)
    print(f"   Saved: {surface_file}")
    
    return surface_file

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def process_all(input_file="rsa_subject_mean.nii.gz", 
                create_highres=True, 
                create_mesh=False,
                smooth_level="medium"):
    """
    Run the complete pipeline
    
    Parameters:
    -----------
    input_file : str
        Your input NIfTI file
    create_highres : bool
        Whether to create high-resolution version
    create_mesh : bool
        Whether to create surface mesh
    smooth_level : str
        "low" (4mm), "medium" (6mm), "high" (8mm), or "ultra" (10mm)
    """
    
    # Set smoothing based on level
    smooth_params = {
        "low": 4.0,
        "medium": 6.0,
        "high": 8.0,
        "ultra": 10.0
    }
    smooth_fwhm = smooth_params.get(smooth_level, 6.0)
    
    print("\n" + "="*60)
    print("SMOOTH BRAIN 3D VISUALIZATION PIPELINE")
    print("="*60)
    print(f"Input file: {input_file}")
    print(f"Smoothing level: {smooth_level} ({smooth_fwhm}mm FWHM)")
    print(f"Create high-res: {create_highres}")
    print(f"Create mesh: {create_mesh}")
    
    # Check if input exists
    import os
    if not os.path.exists(input_file):
        print(f"\nERROR: Input file '{input_file}' not found!")
        print("Please check the filename and path.")
        return
    
    # Process basic smooth brain
    output_smooth = "rsa_smooth_brain.nii.gz"
    try:
        create_smooth_brain_surface(
            input_file,
            output_smooth,
            smooth_fwhm=smooth_fwhm,
            threshold_percentile=20
        )
        print(f"\n✓ Basic smooth brain created: {output_smooth}")
    except Exception as e:
        print(f"\n✗ Failed to create smooth brain: {e}")
        return
    
    # Process high-resolution version
    if create_highres:
        output_highres = "rsa_smooth_brain_highres.nii.gz"
        try:
            create_high_res_smooth_brain(
                input_file,
                output_highres,
                target_resolution=1.5
            )
            print(f"\n✓ High-res smooth brain created: {output_highres}")
        except Exception as e:
            print(f"\n✗ Failed to create high-res version: {e}")
    
    # Create surface mesh
    if create_mesh:
        try:
            surface_file = create_surface_mesh(
                output_smooth,
                "rsa_mesh",
                threshold=0.01
            )
            if surface_file:
                print(f"\n✓ Surface mesh created: {surface_file}")
        except Exception as e:
            print(f"\n✗ Failed to create surface mesh: {e}")
    
    # Final instructions
    print("\n" + "="*60)
    print("PROCESSING COMPLETE!")
    print("="*60)
    print("\nTo visualize in FSLeyes:")
    print("1. Open FSLeyes")
    print(f"2. File → Add from file → Select '{output_smooth}'")
    print("3. Click the '3D' button in the toolbar")
    print("4. In 3D view, adjust the threshold slider (bottom) to fine-tune")
    print("\nTroubleshooting:")
    print("• Still blocky? → Re-run with smooth_level='high' or 'ultra'")
    print("• Too much cut off? → Decrease threshold_percentile to 10-15")
    print("• Want smoother? → Use the highres version (create_highres=True)")

# ============================================================================
# RUN THE PIPELINE
# ============================================================================

# EDIT THESE PARAMETERS AS NEEDED:
INPUT_FILE = "rsa_subject_mean.nii.gz"  # Your input file
SMOOTH_LEVEL = "medium"  # Options: "low", "medium", "high", "ultra"
CREATE_HIGHRES = True    # Create high-resolution version?
CREATE_MESH = False      # Create surface mesh? (requires scikit-image)

# Run the complete pipeline
process_all(
    input_file=INPUT_FILE,
    create_highres=CREATE_HIGHRES,
    create_mesh=CREATE_MESH,
    smooth_level=SMOOTH_LEVEL
)


SMOOTH BRAIN 3D VISUALIZATION PIPELINE
Input file: rsa_subject_mean.nii.gz
Smoothing level: medium (6.0mm FWHM)
Create high-res: True
Create mesh: False

Processing: rsa_subject_mean.nii.gz
1. Loading data...
   Data shape: (53, 63, 46, 575)
2. Calculating threshold...
   Threshold: 0.003584
3. Creating brain mask and smoothing...
   Processing 4D data with 575 timepoints...
   Processed timepoint 575/575
4. Creating smooth brain envelope...
5. Saving smooth brain...
   Saved: rsa_smooth_brain.nii.gz (118.1 MB)

✓ Basic smooth brain created: rsa_smooth_brain.nii.gz

Creating high-resolution smooth brain...
1. Creating initial smooth version...

Processing: rsa_subject_mean.nii.gz
1. Loading data...
   Data shape: (53, 63, 46, 575)
2. Calculating threshold...
   Threshold: 0.002671
3. Creating brain mask and smoothing...
   Processing 4D data with 575 timepoints...
   Processed timepoint 575/575
4. Creating smooth brain envelope...
5. Saving smooth brain...
   Saved: temp_smooth.nii.gz

In [16]:
#!/usr/bin/env python
"""
Simple converter: 4D NIfTI to surface mesh for FSLeyes 3D view
This creates a smooth surface mesh from your data that looks good in FSLeyes
"""

import numpy as np
import nibabel as nib
from scipy.ndimage import gaussian_filter, binary_fill_holes
from skimage import measure
from nibabel import gifti

def nifti_to_surface_mesh(input_file, output_prefix="surface", smooth_mm=4.0):
    """
    Convert 4D NIfTI to surface mesh for FSLeyes 3D visualization
    
    Parameters:
    -----------
    input_file : str
        Path to your 4D NIfTI file
    output_prefix : str
        Prefix for output files
    smooth_mm : float
        Smoothing in mm (higher = smoother surface)
    """
    
    print(f"Loading {input_file}...")
    img = nib.load(input_file)
    data = img.get_fdata()
    print(f"Data shape: {data.shape}")
    
    # Handle 4D data - take temporal mean for surface
    if len(data.shape) == 4:
        print(f"4D data with {data.shape[3]} timepoints detected")
        print("Creating surface from temporal mean...")
        mean_data = np.mean(np.abs(data), axis=3)
        n_timepoints = data.shape[3]
    else:
        mean_data = np.abs(data)
        n_timepoints = None
    
    # Remove NaNs
    mean_data = np.nan_to_num(mean_data, nan=0.0)
    
    # Calculate threshold (remove bottom 30% of values)
    non_zero = mean_data[mean_data > 0]
    if len(non_zero) > 0:
        threshold = np.percentile(non_zero, 30)
    else:
        threshold = 0.001
    
    print(f"Threshold: {threshold:.6f}")
    
    # Apply smoothing for better surface
    print(f"Applying {smooth_mm}mm smoothing...")
    voxel_sizes = nib.affines.voxel_sizes(img.affine)[:3]
    sigma = smooth_mm / (2.355 * np.mean(voxel_sizes))  # Convert FWHM to sigma
    smoothed = gaussian_filter(mean_data, sigma=sigma)
    
    # Create mask
    mask = smoothed > threshold
    mask = binary_fill_holes(mask)
    
    # Apply mask
    smoothed = smoothed * mask
    
    # Extract surface using marching cubes
    print("Extracting surface mesh...")
    verts, faces, normals, values = measure.marching_cubes(
        smoothed,
        level=threshold,
        spacing=voxel_sizes
    )
    
    print(f"Surface extracted: {len(verts):,} vertices, {len(faces):,} faces")
    
    # Transform vertices to world coordinates
    verts_homogeneous = np.column_stack([verts, np.ones(len(verts))])
    verts_world = verts_homogeneous.dot(img.affine.T)[:, :3]
    
    # Create GIFTI surface file
    print("Creating GIFTI surface file...")
    
    # Coordinate array
    coords = gifti.GiftiDataArray(
        data=verts_world.astype(np.float32),
        intent='NIFTI_INTENT_POINTSET',
        datatype='NIFTI_TYPE_FLOAT32',
        coordsys=gifti.GiftiCoordSystem(
            dataspace='NIFTI_XFORM_UNKNOWN',
            xformspace='NIFTI_XFORM_UNKNOWN'
        )
    )
    
    # Triangle array
    triangles = gifti.GiftiDataArray(
        data=faces.astype(np.int32),
        intent='NIFTI_INTENT_TRIANGLE',
        datatype='NIFTI_TYPE_INT32'
    )
    
    # Create and save surface
    surface_gii = gifti.GiftiImage(darrays=[coords, triangles])
    surface_file = f"{output_prefix}.surf.gii"
    nib.save(surface_gii, surface_file)
    print(f"Saved surface: {surface_file}")
    
    # If 4D, create time series overlay
    if n_timepoints:
        print(f"Creating 4D overlay with {n_timepoints} timepoints...")
        
        # For each vertex, sample the time series from nearest voxel
        vertex_timeseries = np.zeros((len(verts), n_timepoints), dtype=np.float32)
        
        for i, vert in enumerate(verts):
            # Get nearest voxel coordinate
            vox_coord = np.round(vert).astype(int)
            vox_coord = np.clip(vox_coord, 0, np.array(data.shape[:3]) - 1)
            
            # Extract time series
            vertex_timeseries[i, :] = data[vox_coord[0], vox_coord[1], vox_coord[2], :]
            
            if i % 1000 == 0:
                print(f"  Sampling vertex {i}/{len(verts)}", end='\r')
        
        print(f"\n  Creating 4D GIFTI overlay...")
        
        # Create GIFTI with time series
        overlay_gii = gifti.GiftiImage()
        
        # Add each timepoint as a data array
        for t in range(n_timepoints):
            tdata = gifti.GiftiDataArray(
                data=vertex_timeseries[:, t],
                intent='NIFTI_INTENT_TIME_SERIES',
                datatype='NIFTI_TYPE_FLOAT32'
            )
            overlay_gii.add_gifti_data_array(tdata)
            
            if t % 50 == 0:
                print(f"  Adding timepoint {t}/{n_timepoints}", end='\r')
        
        overlay_file = f"{output_prefix}.time.gii"
        nib.save(overlay_gii, overlay_file)
        print(f"\nSaved 4D overlay: {overlay_file}")
        
        return surface_file, overlay_file
    
    return surface_file, None


def smooth_nifti_before_mesh(input_file, output_file, smooth_mm=6.0):
    """
    Optional: Smooth the NIfTI file before creating mesh for ultra-smooth surface
    """
    print(f"Pre-smoothing {input_file}...")
    img = nib.load(input_file)
    data = img.get_fdata()
    
    # Apply Gaussian smoothing
    voxel_sizes = nib.affines.voxel_sizes(img.affine)[:3]
    sigma = smooth_mm / (2.355 * np.mean(voxel_sizes))
    
    if len(data.shape) == 4:
        # Smooth each timepoint
        smoothed = np.zeros_like(data)
        for t in range(data.shape[3]):
            smoothed[:, :, :, t] = gaussian_filter(data[:, :, :, t], sigma=sigma)
    else:
        smoothed = gaussian_filter(data, sigma=sigma)
    
    # Save smoothed version
    smooth_img = nib.Nifti1Image(smoothed.astype(np.float32), img.affine, img.header)
    nib.save(smooth_img, output_file)
    print(f"Saved smoothed NIfTI: {output_file}")
    return output_file


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    
    # Your input file
    INPUT_FILE = "rsa_subject_mean.nii.gz"
    
    print("="*60)
    print("4D NIfTI to Surface Mesh Converter")
    print("="*60)
    
    # Option 1: Direct conversion (faster)
    print("\nOption 1: Direct conversion to surface mesh...")
    try:
        surface_file, overlay_file = nifti_to_surface_mesh(
            INPUT_FILE,
            output_prefix="rsa_surface",
            smooth_mm=4.0  # Adjust for smoothness (2-8 mm typical)
        )
        
        print("\n" + "="*60)
        print("SUCCESS! Files created:")
        print("="*60)
        print(f"Surface mesh: {surface_file}")
        if overlay_file:
            print(f"4D overlay: {overlay_file}")
        
        print("\nTo view in FSLeyes:")
        print("1. Open FSLeyes")
        print(f"2. File → Add from file → Select '{surface_file}'")
        print("3. The 3D view should open automatically")
        print("4. If you have 4D data, you can load the overlay too")
        print("\nIn FSLeyes 3D view:")
        print("- Use mouse to rotate")
        print("- Adjust threshold slider at bottom")
        print("- Change colormap for different visualization")
        
    except ImportError:
        print("\nERROR: scikit-image is required for surface extraction")
        print("Install with: pip install scikit-image")
    except Exception as e:
        print(f"\nERROR: {e}")
    
    print("\n" + "="*60)
    
    # Option 2: Ultra-smooth surface (optional)
    print("\nOption 2: Creating ultra-smooth surface (optional)...")
    print("Uncomment the code below if you want an even smoother surface:")
    print("-"*60)
    print("""
    # First smooth the NIfTI heavily
    smoothed_nifti = smooth_nifti_before_mesh(
        INPUT_FILE,
        "rsa_heavily_smoothed.nii.gz",
        smooth_mm=8.0  # Heavy smoothing
    )
    
    # Then create surface from smoothed data
    surface_file, overlay_file = nifti_to_surface_mesh(
        smoothed_nifti,
        output_prefix="rsa_ultrasmooth_surface",
        smooth_mm=2.0  # Light additional smoothing
    )
    """)

4D NIfTI to Surface Mesh Converter

Option 1: Direct conversion to surface mesh...
Loading rsa_subject_mean.nii.gz...
Data shape: (53, 63, 46, 575)
4D data with 575 timepoints detected
Creating surface from temporal mean...
Threshold: 0.008871
Applying 4.0mm smoothing...
Extracting surface mesh...
Surface extracted: 17,001 vertices, 34,008 faces
Creating GIFTI surface file...
Saved surface: rsa_surface.surf.gii
Creating 4D overlay with 575 timepoints...
  Sampling vertex 17000/17001
  Creating 4D GIFTI overlay...
  Adding timepoint 550/575
Saved 4D overlay: rsa_surface.time.gii

SUCCESS! Files created:
Surface mesh: rsa_surface.surf.gii
4D overlay: rsa_surface.time.gii

To view in FSLeyes:
1. Open FSLeyes
2. File → Add from file → Select 'rsa_surface.surf.gii'
3. The 3D view should open automatically
4. If you have 4D data, you can load the overlay too

In FSLeyes 3D view:
- Use mouse to rotate
- Adjust threshold slider at bottom
- Change colormap for different visualization


Option 2

# Resample only

In [17]:
import numpy as np
import nibabel as nib
from nilearn.image import resample_img

# Load your file
input_file = "rsa_subject_mean.nii.gz"
output_file = "rsa_subject_mean_1mm.nii.gz"

print(f"Loading {input_file}...")
img = nib.load(input_file)

# Get current voxel sizes
current_voxel_sizes = nib.affines.voxel_sizes(img.affine)[:3]
print(f"Current voxel sizes: {current_voxel_sizes} mm")
print(f"Current shape: {img.shape}")

# Calculate new affine for 1mm voxels
target_resolution = 1.0  # 1mm
scale_factors = current_voxel_sizes / target_resolution

# Create new affine matrix
target_affine = img.affine.copy()
target_affine[:3, :3] = img.affine[:3, :3] / scale_factors[:, np.newaxis]

# Calculate new shape to maintain the same FOV
current_shape = img.shape[:3]
target_shape = tuple(int(np.round(s * sf)) for s, sf in zip(current_shape, scale_factors))

print(f"Target voxel size: [1.0, 1.0, 1.0] mm")
print(f"Target shape: {target_shape}")

# Resample
print("Resampling to 1mm resolution...")
resampled = resample_img(
    img,
    target_affine=target_affine,
    target_shape=target_shape,
    interpolation='continuous'
)

# Save
print(f"Saving to {output_file}...")
nib.save(resampled, output_file)

# Report file sizes
import os
original_size = os.path.getsize(input_file) / (1024 * 1024)  # MB
new_size = os.path.getsize(output_file) / (1024 * 1024)  # MB

print(f"\nDone!")
print(f"Original file: {original_size:.1f} MB")
print(f"Resampled file: {new_size:.1f} MB")
print(f"Output saved as: {output_file}")

Loading rsa_subject_mean.nii.gz...
Current voxel sizes: [3. 3. 3.] mm
Current shape: (53, 63, 46, 575)
Target voxel size: [1.0, 1.0, 1.0] mm
Target shape: (159, 189, 138)
Resampling to 1mm resolution...
Saving to rsa_subject_mean_1mm.nii.gz...

Done!
Original file: 120.2 MB
Resampled file: 3175.4 MB
Output saved as: rsa_subject_mean_1mm.nii.gz


In [18]:
import numpy as np
import nibabel as nib

# Load file
img = nib.load("rsa_subject_mean_1mm.nii.gz")
data = img.get_fdata()

# Set near-zero values to NaN (FSLeyes ignores NaN in 3D view)
threshold = 0.001
data[np.abs(data) < threshold] = np.nan

# Save
new_img = nib.Nifti1Image(data.astype(np.float32), img.affine, img.header)
nib.save(new_img, "rsa_subject_mean_1mm_no_box.nii.gz")

# Surface

In [21]:
import numpy as np
import nibabel as nib
from nibabel import gifti
from skimage import measure

def create_fsleyes_compatible_4d_surface(input_4d_nifti, output_prefix):
    """
    Create FSLeyes-compatible 4D surface files
    """
    print("Loading 4D data...")
    img = nib.load(input_4d_nifti)
    data = img.get_fdata()
    print(f"Shape: {data.shape}")
    
    # Create surface from temporal mean
    mean_vol = np.mean(np.abs(data), axis=3)
    threshold = np.percentile(mean_vol[mean_vol > 0], 20)
    
    # Extract surface mesh
    print("Creating surface mesh...")
    verts, faces, _, _ = measure.marching_cubes(
        mean_vol,
        level=threshold,
        spacing=nib.affines.voxel_sizes(img.affine)[:3]
    )
    
    # Transform to world coordinates
    verts_hom = np.column_stack([verts, np.ones(len(verts))])
    verts_world = verts_hom.dot(img.affine.T)[:, :3]
    
    # === SURFACE FILE (geometry only) ===
    print("Creating surface geometry file...")
    
    # Create coordinate array
    coords = gifti.GiftiDataArray(
        data=verts_world.astype(np.float32),
        intent='NIFTI_INTENT_POINTSET',
        datatype='NIFTI_TYPE_FLOAT32'
    )
    
    # Create triangle array - MUST be exactly one
    triangles = gifti.GiftiDataArray(
        data=faces.astype(np.int32),
        intent='NIFTI_INTENT_TRIANGLE',
        datatype='NIFTI_TYPE_INT32'
    )
    
    # Create surface GIFTI with ONLY geometry
    surface_gii = gifti.GiftiImage()
    surface_gii.add_gifti_data_array(coords)
    surface_gii.add_gifti_data_array(triangles)
    
    surface_file = f"{output_prefix}_surface.surf.gii"
    nib.save(surface_gii, surface_file)
    print(f"Saved surface: {surface_file}")
    
    # === SEPARATE FILE for 4D DATA ===
    print("Creating 4D overlay file...")
    n_timepoints = data.shape[3]
    vertex_data = np.zeros((len(verts), n_timepoints))
    
    # Sample time series at each vertex
    for i, vert in enumerate(verts):
        vox = np.round(vert).astype(int)
        vox = np.clip(vox, 0, np.array(data.shape[:3]) - 1)
        vertex_data[i, :] = data[vox[0], vox[1], vox[2], :]
    
    # Create separate GIFTI for time series data
    timeseries_gii = gifti.GiftiImage()
    
    # Add metadata to indicate this is time series data
    timeseries_gii.meta = gifti.GiftiMetaData()
    timeseries_gii.meta.data.append(
        gifti.GiftiNVPairs(name='TimeStep', value='1.0')
    )
    
    # Add all timepoints
    for t in range(n_timepoints):
        tdata = gifti.GiftiDataArray(
            data=vertex_data[:, t].astype(np.float32),
            intent='NIFTI_INTENT_NONE',  # Changed from TIME_SERIES
            datatype='NIFTI_TYPE_FLOAT32'
        )
        timeseries_gii.add_gifti_data_array(tdata)
    
    timeseries_file = f"{output_prefix}_timeseries.func.gii"
    nib.save(timeseries_gii, timeseries_file)
    print(f"Saved timeseries: {timeseries_file}")
    
    print("\n" + "="*60)
    print("FILES CREATED SUCCESSFULLY")
    print("="*60)
    print("\nTo load in FSLeyes:")
    print(f"1. File → Add from file → {surface_file}")
    print(f"2. Then: File → Add from file → {timeseries_file}")
    print("\nThe surface will load first, then the time series will be applied to it.")
    
    return surface_file, timeseries_file

# Run it
create_fsleyes_compatible_4d_surface(
    "rsa_subject_mean_1mm.nii.gz",
    "rsa_fsleyes"
)

Loading 4D data...
Shape: (159, 189, 138, 575)
Creating surface mesh...
Creating surface geometry file...
Saved surface: rsa_fsleyes_surface.surf.gii
Creating 4D overlay file...
Saved timeseries: rsa_fsleyes_timeseries.func.gii

FILES CREATED SUCCESSFULLY

To load in FSLeyes:
1. File → Add from file → rsa_fsleyes_surface.surf.gii
2. Then: File → Add from file → rsa_fsleyes_timeseries.func.gii

The surface will load first, then the time series will be applied to it.


('rsa_fsleyes_surface.surf.gii', 'rsa_fsleyes_timeseries.func.gii')

In [23]:
import nibabel as nib
import numpy as np

# Paths
p = "N:/Experimental_Data/yujunchen/projects/IAPS_EEG_RSA/fs_rsa/"
in_file = p + "rsa_tp255.nii.gz"
out_file = p + "rsa_tp255_nonan.nii.gz"

# Load
img = nib.load(in_file)
data = img.get_fdata()

# Replace NaNs with 0
data = np.nan_to_num(data, nan=0.0)

# Save back as float32
out = nib.Nifti1Image(data.astype(np.float32), img.affine, img.header)
nib.save(out, out_file)

print("Wrote", out_file)


Wrote N:/Experimental_Data/yujunchen/projects/IAPS_EEG_RSA/fs_rsa/rsa_tp255_nonan.nii.gz


In [33]:
from pathlib import Path
import nibabel as nib

work = Path(r"N:\Experimental_Data\yujunchen\projects\IAPS_EEG_RSA\fs_rsa")

def merge(pattern, outname):
    files = sorted(work.glob(pattern))
    print(f"{len(files)} files -> {outname}")
    first = nib.load(str(files[0]))
    darrays = [nib.gifti.GiftiDataArray(nib.load(str(f)).darrays[0].data.astype('float32'))
               for f in files]
    out = nib.gifti.GiftiImage(darrays=darrays, meta=first.meta, labeltable=first.labeltable)
    nib.save(out, str(work/outname))

merge("lh.tp*.func.gii", "lh.rsa_all.func.gii")
merge("rh.tp*.func.gii", "rh.rsa_all.func.gii")


574 files -> lh.rsa_all.func.gii
574 files -> rh.rsa_all.func.gii


In [35]:
import nibabel as nib
import numpy as np
from pathlib import Path

p = Path(r"N:\Experimental_Data\yujunchen\projects\IAPS_EEG_RSA")
in_file  = p / "rsa_subject_mean.nii.gz"
out_file = p / "rsa_subject_mean_nonan.nii.gz"

img  = nib.load(str(in_file))
data = img.get_fdata()                 # loads full 4D array into RAM
data = np.nan_to_num(data, nan=0.0)    # NaN -> 0

out = nib.Nifti1Image(data.astype(np.float32), img.affine, img.header)
out.header.set_data_dtype(np.float32)   # keep file size reasonable
nib.save(out, str(out_file))
print("Wrote", out_file)


Wrote N:\Experimental_Data\yujunchen\projects\IAPS_EEG_RSA\rsa_subject_mean_nonan.nii.gz


In [44]:
from pathlib import Path
import re
import nibabel as nib
from nibabel.gifti import GiftiImage, GiftiDataArray
from nibabel.nifti1 import intent_codes  # provides NIFTI intent codes

work = Path(r"N:\Experimental_Data\yujunchen\projects\IAPS_EEG_RSA\fs_rsa_all")

# Natural sort by the numeric frame index (tp000, tp001, ...)
def natural_key(p: Path):
    m = re.search(r'\.tp(\d+)\.', p.name)
    return int(m.group(1)) if m else p.name

def combine_dataonly(pattern: str, outname: str):
    files = sorted(work.glob(pattern), key=natural_key)
    if not files:
        raise FileNotFoundError(f"No files match {work / pattern}")

    scalars = []
    nverts = None

    for f in files:
        img = nib.load(str(f))

        # Expect exactly one scalar darray (no geometry in data-only files)
        darrays = [da for da in img.darrays
                   if da.intent not in (intent_codes['NIFTI_INTENT_POINTSET'],
                                        intent_codes['NIFTI_INTENT_TRIANGLE'])]
        if len(darrays) != 1:
            raise ValueError(f"{f.name}: expected 1 scalar array, found {len(darrays)}")

        arr = darrays[0].data
        if nverts is None:
            nverts = arr.shape[0]
        elif arr.shape[0] != nverts:
            raise ValueError(f"Vertex count mismatch in {f.name}: {arr.shape[0]} vs {nverts}")

        # Re-wrap as float32 SHAPE intent; keep this darray metadata minimal
        scalars.append(GiftiDataArray(arr.astype('float32'),
                                      intent=intent_codes['NIFTI_INTENT_SHAPE']))

    out = GiftiImage(darrays=scalars)
    nib.save(out, str(work / outname))
    print(f"Wrote {outname} with {len(scalars)} frames (nverts={nverts})")

# --- Combine LH and RH ---
combine_dataonly("lh.tp*.inflated.func.dataonly.func.gii", "lh.dataonly.multiframe.func.gii")
combine_dataonly("rh.tp*.inflated.func.dataonly.func.gii", "rh.dataonly.multiframe.func.gii")


Wrote lh.dataonly.multiframe.func.gii with 574 frames (nverts=163842)
Wrote rh.dataonly.multiframe.func.gii with 574 frames (nverts=163842)
